In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv(
    "../data/processed/feature_engineered_shots.csv"
)

df.head()

,minute,second,period,team_name,player_name,position_name,x,y,end_x,end_y,...,shot_one_on_one,shot_open_goal,shot_deflected,shot_statsbomb_xg,outcome_name,season_name,goal,shot_distance,angle,centrality
0,0,18,1,Switzerland,Granit Xhaka,Left Defensive Midfield,96.0,38.8,108.2,38.5,...,False,False,False,0.036566,Blocked,2022,0,24.029981,0.301942,1.2
1,0,22,1,Switzerland,Breel-Donald Embolo,Center Forward,113.1,40.7,114.8,40.6,...,False,False,False,0.353289,Saved,2022,0,6.935416,0.968776,0.7
2,0,23,1,Switzerland,Granit Xhaka,Left Defensive Midfield,103.8,41.9,115.5,39.1,...,False,False,False,0.069527,Saved,2022,0,16.311039,0.438830,1.9
3,4,35,1,Serbia,Nikola Milenković,Right Center Back,112.2,36.8,120.0,35.3,...,False,False,False,0.081609,Off T,2022,0,8.430896,0.780272,3.2
4,10,5,1,Serbia,Andrija Živković,Right Wing Back,97.8,51.5,120.0,36.1,...,False,False,False,0.030002,Post,2022,0,25.001800,0.259664,11.5


In [3]:
features = [
    'minute',
    'period',
    'x',
    'y',
    'shot_distance',
    'angle',
    'centrality',
    'body_part_name',
    'technique_name',
    'play_pattern_name',
    'under_pressure',
    'shot_first_time',
    'shot_one_on_one',
    'shot_open_goal',
    'shot_deflected'
]

X = df[features]

y = df['goal']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(2560, 15)
(640, 15)


In [5]:
numerical_features = [
    'minute',
    'period',
    'x',
    'y',
    'shot_distance',
    'angle',
    'centrality'
]

categorical_features = [
    'body_part_name',
    'technique_name',
    'play_pattern_name',
    'under_pressure',
    'shot_first_time',
    'shot_one_on_one',
    'shot_open_goal',
    'shot_deflected'
]

In [6]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [7]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [10]:
# logistic regression pipeline
logreg_pipeline = Pipeline(
    steps = [
        ('preprocessor',preprocessor),
        ('model',LogisticRegression(max_iter=1000))
    ]
) 

In [11]:
logreg_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['minute', 'period', 'x', 'y',
                                                   'shot_distance', 'angle',
                                                   'centrality']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['body_part_name',
                                                   'technique_name',
                                                   'play_pattern_name',
                                                   'under_pressure',
                                                   'shot_first_time',
                                                   'shot_one_on_one',
                                                   'shot_open_goal',
                                                   'shot_deflected'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [14]:
y_pred_logreg = logreg_pipeline.predict(X_test)

y_prob_logreg = logreg_pipeline.predict_proba(X_test)[:,1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test,y_pred_logreg))
print("Precision:", precision_score(y_test,y_pred_logreg))
print("Recall:", recall_score(y_test,y_pred_logreg))
print("F1 Score:", f1_score(y_test,y_pred_logreg))
print("ROC-AUC:", roc_auc_score(y_test,y_prob_logreg))

Logistic Regression
Accuracy: 0.8953125
Precision: 0.6046511627906976
Recall: 0.34210526315789475
F1 Score: 0.4369747899159664
ROC-AUC: 0.8113801791713326


In [27]:
# random forest classifier pipeline
rf_pipeline = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators = 200,
            random_state = 42
        ))
    ]
)

In [28]:
rf_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['minute', 'period', 'x', 'y',
                                                   'shot_distance', 'angle',
                                                   'centrality']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['body_part_name',
                                                   'technique_name',
                                                   'play_pattern_name',
                                                   'under_pressure',
                                                   'shot_first_time',
                                                   'shot_one_on_one',
                                                   'shot_open_goal',
                                                   'shot_deflected'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [29]:
y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:,1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test,y_pred_rf))
print("Precision:", precision_score(y_test,y_pred_rf))
print("Recall:", recall_score(y_test,y_pred_rf))
print("F1 Score:", f1_score(y_test,y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test,y_pred_rf))

Random Forest
Accuracy: 0.896875
Precision: 0.6136363636363636
Recall: 0.35526315789473684
F1 Score: 0.45
ROC-AUC: 0.6625606569615528


In [30]:
# XG Boost pipeline
xgb_pipeline = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model', XGBClassifier(
            n_estimators = 200,
            learning_rate = 0.05,
            max_depth = 4,
            random_state= 42,
            eval_metric = 'logloss'
        ))
    ]
)

In [31]:
xgb_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['minute', 'period', 'x', 'y',
                                                   'shot_distance', 'angle',
                                                   'centrality']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [33]:
y_pred_xgb = xgb_pipeline.predict(X_test)
y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:,1]

print("XGBoost")
print("Accuracy:", accuracy_score(y_test,y_pred_xgb))
print("Precision:", precision_score(y_test,y_pred_xgb))
print("Recall:", recall_score(y_test,y_pred_xgb))
print("F1 Score:", f1_score(y_test,y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test,y_pred_xgb))


XGBoost
Accuracy: 0.896875
Precision: 0.6388888888888888
Recall: 0.3026315789473684
F1 Score: 0.4107142857142857
ROC-AUC: 0.6397909667786487
